<a href="https://colab.research.google.com/github/CarlosEduDev/Natural-Language-Processing-with-Disaster-Tweets/blob/main/Natural_Language_Processing_with_Disaster_Tweets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Natural Language Processing with Disaster Tweets

In [148]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

In [149]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

In [150]:
def clean_text(text):
  text = text.lower()
  text = re.sub(r'https?://\S+|www\.\S+', '', text)  # Remove URLs
  text = re.sub(r'<.*?>+', '', text)                 # Remove tags HTML
  text = re.sub(r'[^a-zA-Z0-9\s]', '', text)            # Mantem apenas letras e espacos
  return text

train_df['text_clean'] = train_df['text'].apply(clean_text)
test_df['text_clean'] = test_df['text'].apply(clean_text)

In [151]:
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), sublinear_tf=True)
X_train = vectorizer.fit_transform(train_df['text_clean'])
y_train = train_df['target']

In [152]:
# treinamento e avaliação inicial (validação cruzada)
model = LogisticRegression(C = 1.0, max_iter = 5000)
scores = cross_val_score(model, X_train, y_train, cv = 5, scoring = 'f1')

print(f"F1-Score médio na avaliação cruzada {scores.mean():.4f}")

F1-Score médio na avaliação cruzada 0.6228


In [153]:
# Treinamento com a base completa e geracao do arquivo de submissao
model.fit(X_train, y_train)
prediction = model.predict(X_test)

submission = pd.DataFrame({
    'id': test_df['id'],
    'target': prediction
    })

submission.to_csv("submission_baseline.csv", index = False)